<a href="https://colab.research.google.com/github/Digital-AI-Finance/Cryptocurrency/blob/main/02_cryptography_security/notebooks/NB02_SHA256_Avalanche_Interactive.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# SHA-256 Avalanche Effect: Interactive Explorer

**Course:** Build Your Own Cryptocurrency  
**Lesson:** 2 - Cryptography & Security

## Learning Objectives
By the end of this notebook, you will:
- Understand the SHA-256 avalanche effect and why it matters for blockchain security
- Observe how minimal input changes produce drastically different hash outputs
- Quantify bit-level differences between hash outputs
- Experiment with your own inputs to verify the avalanche property

In [ ]:
# Setup
!pip install -q matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, Rectangle
import hashlib
print("Setup complete!")

## What is the Avalanche Effect?

The **avalanche effect** is a property of cryptographic hash functions where a tiny change in the input (even flipping a single bit) causes a dramatic, unpredictable change in the output. Ideally, changing one bit of input should flip approximately 50% of the output bits.

This property is crucial for blockchain security because:
- It makes it impossible to predict how a hash will change
- It prevents attackers from finding collisions efficiently
- It ensures that even similar transactions produce completely different hashes

In [ ]:
def hash_to_binary(text):
    """Convert text to SHA-256 hash and return hex and binary representations."""
    hash_hex = hashlib.sha256(text.encode()).hexdigest()
    hash_bin = bin(int(hash_hex, 16))[2:].zfill(256)
    return hash_hex, hash_bin

# Quick test
test_hex, test_bin = hash_to_binary("test")
print(f"Input: 'test'")
print(f"SHA-256 (hex): {test_hex}")
print(f"SHA-256 (bin): {test_bin[:64]}... ({len(test_bin)} bits total)")

## Comparing Two Similar Inputs

Below, we define two input strings that differ by only **one character** (uppercase vs lowercase first letter). Change these to experiment!

In [ ]:
# ========================================
# CHANGE THESE TO EXPERIMENT!
# ========================================
input1 = "Bitcoin"
input2 = "bitcoin"   # Only difference: capital 'B' vs lowercase 'b'
# ========================================

# Compute hashes
hash1_hex, hash1_bin = hash_to_binary(input1)
hash2_hex, hash2_bin = hash_to_binary(input2)

print(f"Input 1: '{input1}'")
print(f"Hash 1:  {hash1_hex}")
print(f"\nInput 2: '{input2}'")
print(f"Hash 2:  {hash2_hex}")

# Count differing bits
differences = sum(1 for i in range(256) if hash1_bin[i] != hash2_bin[i])
percentage = (differences / 256) * 100
print(f"\nBit differences: {differences} out of 256 ({percentage:.1f}%)")
print(f"Expected for ideal hash: ~128 bits (50%)")

## Visual Bit Comparison

The visualization below shows the first 64 bits of each hash as colored blocks. Red outlines highlight where the bits differ between the two hashes.

In [ ]:
# Color palette
MLBLUE = '#0066CC'
MLORANGE = '#FF7F0E'
MLRED = '#D62728'

fig, ax = plt.subplots(figsize=(12, 7))
ax.set_xlim(0, 10)
ax.set_ylim(0, 6)
ax.axis('off')

# Title
ax.text(5, 5.7, 'SHA-256 Avalanche Effect',
        fontsize=20, fontweight='bold', ha='center', va='top')
ax.text(5, 5.4, f'"{input1}" vs "{input2}"',
        fontsize=12, ha='center', va='top', style='italic', color=MLRED)

# Input/Hash boxes
for i, (inp, hsh, y_offset) in enumerate([
    (input1, hash1_hex, 0), (input2, hash2_hex, -1.6)
]):
    # Input box
    input_box = FancyBboxPatch((0.2, 4.2 + y_offset), 4.3, 0.6,
                               boxstyle="round,pad=0.05",
                               edgecolor=MLBLUE, facecolor='lightblue', linewidth=2)
    ax.add_patch(input_box)
    ax.text(0.4, 4.65 + y_offset, f'Input {i+1}:', fontsize=10, fontweight='bold', va='center')
    ax.text(0.4, 4.35 + y_offset, f'"{inp}"', fontsize=11, va='center', family='monospace')

    # Hash box
    hash_box = FancyBboxPatch((5.0, 4.2 + y_offset), 4.8, 0.6,
                               boxstyle="round,pad=0.05",
                               edgecolor='#2CA02C', facecolor='lightgreen', linewidth=2)
    ax.add_patch(hash_box)
    ax.text(5.2, 4.65 + y_offset, f'Hash {i+1}:', fontsize=10, fontweight='bold', va='center')
    ax.text(5.2, 4.35 + y_offset, hsh[:32] + '...', fontsize=8, va='center', family='monospace')

# Binary comparison
ax.text(5, 2.4, 'Binary Bit Comparison (first 64 bits)', fontsize=11, ha='center', fontweight='bold')

block_size = 0.12
start_x = 0.5

for i in range(64):
    x = start_x + i * block_size
    bit1 = hash1_bin[i]
    bit2 = hash2_bin[i]

    color1 = MLBLUE if bit1 == '1' else 'white'
    rect1 = Rectangle((x, 1.8), block_size * 0.9, 0.3,
                       facecolor=color1, edgecolor='black', linewidth=0.5)
    ax.add_patch(rect1)

    color2 = MLORANGE if bit2 == '1' else 'white'
    rect2 = Rectangle((x, 1.3), block_size * 0.9, 0.3,
                       facecolor=color2, edgecolor='black', linewidth=0.5)
    ax.add_patch(rect2)

    if bit1 != bit2:
        highlight = Rectangle((x, 1.25), block_size * 0.9, 0.4,
                              facecolor='none', edgecolor=MLRED, linewidth=2)
        ax.add_patch(highlight)

ax.text(0.3, 1.95, 'Hash 1:', fontsize=9, va='center', fontweight='bold')
ax.text(0.3, 1.45, 'Hash 2:', fontsize=9, va='center', fontweight='bold')

# Statistics
stats_text = f'Bit Differences: {differences} out of 256 bits ({percentage:.1f}%)\nRed boxes show where bits differ'
ax.text(5, 0.7, stats_text, fontsize=10, ha='center', va='center',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

## Understanding the Results

- **~50% bit difference** is expected for a good cryptographic hash function
- Even changing just one character (e.g., uppercase to lowercase) produces a completely different hash
- The output looks random and unpredictable - there's no way to "guess" how the hash will change
- This property makes SHA-256 suitable for securing blockchain transactions

In [ ]:
# Let's test the avalanche effect with multiple input pairs
print("=" * 70)
print("AVALANCHE EFFECT: BATCH COMPARISON")
print("=" * 70)

test_pairs = [
    ("Bitcoin", "bitcoin"),
    ("Hello", "Hellp"),
    ("Block 1", "Block 2"),
    ("Transfer 10 BTC", "Transfer 11 BTC"),
    ("Same input", "Same input"),  # Identical inputs
]

print(f"\n{'Input 1':<20} {'Input 2':<20} {'Bits Changed':>15} {'Percentage':>12}")
print("-" * 70)

for text1, text2 in test_pairs:
    _, bin1 = hash_to_binary(text1)
    _, bin2 = hash_to_binary(text2)
    diff = sum(1 for i in range(256) if bin1[i] != bin2[i])
    pct = (diff / 256) * 100
    print(f"{text1:<20} {text2:<20} {diff:>10}/256  {pct:>10.1f}%")

print("\nNotice: Different inputs always differ by ~50% of bits")
print("Identical inputs produce 0 differences (deterministic)")

## Exercises

Try these experiments by modifying the `input1` and `input2` variables in the cell above:

1. **Your name test**: Set `input1` to your full name and `input2` to your name with one letter capitalized differently. How many bits differ?

2. **Long vs short**: Compare a very long string (100+ characters) with the same string plus one extra character. Is the percentage still ~50%?

3. **Completely different inputs**: Compare "Bitcoin" with "Ethereum". More or fewer bit differences than "Bitcoin" vs "bitcoin"?

4. **Identical inputs**: What happens when both inputs are exactly the same?

5. **Empty string**: What is the SHA-256 hash of an empty string ""? Is it all zeros?

## Key Takeaways

1. **The avalanche effect** ensures that even a 1-bit input change flips ~50% of output bits
2. **This property is essential** for blockchain security - it prevents attackers from predicting hash changes
3. **SHA-256 is deterministic** - the same input always produces the same hash
4. **Hash outputs appear random** but are completely reproducible
5. **No reverse engineering** is possible - you cannot derive the input from the hash
6. **This is why mining works** - changing the nonce slightly produces a completely different hash, making brute-force the only approach